In [4]:
# 1. 开启交互式绘图模式 (关键！)
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

# 1. 设置字体为 Mac 自带的 "Heiti TC" (黑体) 或者 "Arial Unicode MS"
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] 

# 2. 解决负号显示为方块的问题 (这一步对 PCA 降维图很重要，因为坐标轴可能有负数)
plt.rcParams['axes.unicode_minus'] = False


# 2. 加载 MNIST 数据
print("正在下载/加载 MNIST 数据，请稍候...")
X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False, parser='auto')

# 3. 数据切分 (这次我们只需要训练集做实验)
# 只要 60000 张训练图即可，不需要测试集
X_train = X_mnist[:60000]
y_train = y_mnist[:60000]

# 4. 数据预处理 (归一化)
# PCA 对缩放非常敏感，必须把 0-255 变成 0-1
print("正在归一化数据...")
X_train_scaled = X_train / 255.0

print(f"✅ 准备就绪！数据形状: {X_train_scaled.shape}")

正在下载/加载 MNIST 数据，请稍候...
正在归一化数据...
✅ 准备就绪！数据形状: (60000, 784)


In [5]:
# 1. 实例化一个不限制维度的 PCA
print("正在分析主成分方差（这可能需要几秒钟）...")
pca = PCA()
pca.fit(X_train_scaled)

# 2. 计算累计方差
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1  # 找到达到 95% 的那个临界点

# 3. 画图
plt.figure(figsize=(8, 5))
plt.plot(cumsum, linewidth=3)
plt.axis([0, 400, 0, 1])
plt.xlabel("Dimensions (特征数量)")
plt.ylabel("Explained Variance (信息保留率)")
plt.plot([d, d], [0, 0.95], "k:")  # 竖虚线
plt.plot([0, d], [0.95, 0.95], "k:")  # 横虚线
plt.plot(d, 0.95, "ro")  # 红点标记
plt.grid(True)
plt.title(f"PCA Analysis: Need {d} dimensions for 95% variance")
plt.show()

print(f"结论：要把 784 维压缩并保留 95% 的信息，你需要 {d} 维。")

正在分析主成分方差（这可能需要几秒钟）...
结论：要把 784 维压缩并保留 95% 的信息，你需要 154 维。


In [6]:
# 为了绘图流畅，我们只取前 3000 个样本
X_subset = X_train_scaled[:3000]
y_subset = y_train[:3000].astype(int)

print("正在执行 3D 降维 (784 -> 3)...")
pca_3d = PCA(n_components=3)
X_reduced_3d = pca_3d.fit_transform(X_subset)

print("正在生成 3D 交互图 (请尝试用鼠标拖动)...")
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    X_reduced_3d[:, 0], 
    X_reduced_3d[:, 1], 
    X_reduced_3d[:, 2], 
    c=y_subset, 
    cmap='jet',  # 颜色表
    s=10,        # 点的大小
    alpha=0.6    # 透明度
)

ax.set_xlabel('PC 1 (主要特征)')
ax.set_ylabel('PC 2 (次要特征)')
ax.set_zlabel('PC 3 (第三特征)')
ax.set_title('MNIST 3D Visualization')
plt.colorbar(scatter, label='Digit Label')

plt.show()


正在执行 3D 降维 (784 -> 3)...
正在生成 3D 交互图 (请尝试用鼠标拖动)...
